In [1]:
import os
import re
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import polars as pl
import loguru as logger

from settings.settings_analyze_efizz import Settings_ae
from behave_analysis.process.session import get_experiment
from behave_analysis.utils.rayleigh.load_rayleigh import collect_all_rayleigh_paths, load_all_rayleigh_data
from behave_analysis.utils.rayleigh.manipulate_rayleigh_df import extract_compartment_values, extract_firing_rates
from behave_analysis.utils.creating_directories import make_directory
from settings.settings_analyze_efizz import Settings_ae as Settings
from behave_analysis.analyze.TunED.model import TunEdModel

In [2]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept
from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept
from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept
from behave_analysis.database.Experiments.JAL006_ex import JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr
from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr, JAL7_30apr
from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_tiny_3may, JAL8_flip4_10may, JAL8_14may, JAL8_21may

In [3]:
experiments_objects = [JAL6_flip7_1apr, JAL6_flip3_18mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_28mar,
                       JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept,
                       JAL005_8thSept, JAL005_21stSept,
                       JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr,
                       JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may,
                       JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept]

tinny_barrier = [JAL8_tiny_3may, JAL8_21may, JAL7_30apr]

# Mice groups based on session names
mice_groups = {
    "JAL6": ['JAL6_flip7_1apr', 'JAL6_flip3_18mar', 'JAL6_flip4_21mar', 'JAL6_flip5_25mar', 'JAL6_28mar'],
    "JAL3": ['JAL3_25aug', 'JAL3_1sept', 'JAL3_4sept', 'JAL3_7sept'],
    "JAL7": ['JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL7_flip5_22mar', 'JAL7_flip2_12mar', 'JAL7_23apr'],
    "JAL8": ['JAL8_flip1_25apr', 'JAL8_flip2_29apr', 'JAL8_flip4_10may', 'JAL8_14may'],
    "JAL4": ['JAL4_3rdSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_11thSept'],
    "JAL5": ['JAL5_8thSept', 'JAL5_21stSept']}


session_names = ["JAL6_flip7_1apr", "JAL6_flip3_18mar", "JAL6_flip4_21mar", "JAL6_flip5_25mar", "JAL6_28mar",
                 "JAL3_25aug", "JAL3_1sept", "JAL3_4sept", "JAL3_7sept", "JAL005_8thSept", "JAL005_21stSept",
                 "JAL5_8thSept", "JAL5_21stSept",
                 "JAL7_sesh8_9apr", "JAL7_sesh9_16apr", "JAL7_flip5_22mar", "JAL7_flip2_12mar", "JAL7_23apr",
                 "JAL8_flip1_25apr", "JAL8_flip2_29apr", "JAL8_flip4_10may", "JAL8_14may",
                 "JAL4_3rdSept", "JAL4_19thSept", "JAL4_28aug", "JAL4_11thSept"]

In [4]:
def regex(angle):
    "Remove unwanted characters from angle file string"
    pattern = r'^(.*?)(?=_Rayleigh)'
    match = re.search(pattern, angle)
    assert match, f"Could not find match for {angle}"
    return match.group(0)

def nest_dic():
    """A function to create arbitrarily nested dictionaries"""
    return defaultdict(nest_dic)

In [5]:
# Init Params
conditions = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
total_cells = 0
total_sessions = 0
rayleigh_threshold = 0.15
fr_threshold = 5 # Hz
dir = make_directory(r"Z:\Jasmine_Laurence\rayleigh_analysis")

In [6]:
angle_keys = ['hdir_Rayleigh.arrow', 'hsa_Rayleigh.arrow', 'h_postflipbar_a_Rayleigh.arrow', 'h_preflipbar_a_Rayleigh.arrow']

# Just threat zone

In [7]:
# TODO firing rate threshold not implemented

threat_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                    rayleigh = output[cell][1]
                    max_angle_str = regex(angle)
                    tuned = True
                    
                # Just the shelter
                # output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for SHELTER ONLY
                # if np.logical_and(output[cell][0] > rayleigh, output[cell][0] > rayleigh_threshold):
                #     rayleigh = output[cell][0]
                #     max_angle_str = regex(angle)
                #     tuned = True

                # # Whole arena rayleigh
                # output = condition_data[condition][angle]["arena_rayleigh"] # Whole arena rayleigh
                # if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold):
                #     rayleigh = output[cell]
                #     max_angle_str = regex(angle)
                #     tuned = True
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    threat_dict[i][cell][condition] = max_angle_str
                else:
                    threat_dict[i][cell][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1
    

In [8]:
print(not_meet_threshold)
print(cell_count)

1625
4858


# Count across sessions and mice how many cells are tuned to a combo in threat compartment

In [10]:
# Counts across sessions
TC_shelter = []
TC_barrier_pre_flip = []
TC_barrier_post_flip = []
for session in threat_dict:
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        TC_shelter.append(x)
        y = threat_dict[session][cell]["barrier_pre_flip"]
        TC_barrier_pre_flip.append(y)
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_barrier_post_flip.append(z)

TC_shelter_counts = Counter(TC_shelter)
TC_barrier_pre_flip_counts = Counter(TC_barrier_pre_flip)
TC_barrier_post_flip_counts = Counter(TC_barrier_post_flip)
print("Across session counts")
print(TC_shelter_counts)
print(TC_barrier_pre_flip_counts)

TC_shelter_total = 0
TC_barrier_pre_flip_total = 0
TC_barrier_post_flip_total = 0
c1 = sum([i for i in TC_shelter_counts.values()])
c2 = sum([i for i in TC_barrier_pre_flip_counts.values()])
c3 = sum([i for i in TC_barrier_post_flip_counts.values()])
print(f"Cells in shelter_only: {c1}")
print(f"Cells in barrier_pre_flip: {c2}")
print(f"Cells in barrier_post_flip: {c3}")
print(f"Total cells: {cell_count}")

Across session counts
Counter({'Not tuned': 2107, 'h_preflipbar_a': 802, 'h_postflipbar_a': 764, 'hdir': 623, 'hsa': 562})
Counter({'Not tuned': 1752, 'h_preflipbar_a': 970, 'hdir': 894, 'h_postflipbar_a': 673, 'hsa': 569})
Cells in shelter_only: 4858
Cells in barrier_pre_flip: 4858
Cells in barrier_post_flip: 4858
Total cells: 4858


# count within session

In [10]:
# Counts within sessions
TC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in threat_dict:
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        y = threat_dict[session][cell]["barrier_pre_flip"]
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_within_session_counts_shelter[session][x] += 1
        TC_within_session_counts_bar_pre_flip[session][y] += 1
        TC_within_session_counts_bar_post_flip[session][z] += 1

# Just shelter compartment

In [11]:
# TODO firing rate threshold not implemented

shelter_dict = nest_dic() # dict[session][cell][condition] = {max_rayleigh_angle}
cell_count = 0
not_meet_threshold = 0

for i, session in enumerate(experiments_objects):
    loaded_session = get_experiment(session)
    paths = collect_all_rayleigh_paths(session = loaded_session, cluster_type = "good", conditions= conditions) # paths[condition][angles]
    condition_data = load_all_rayleigh_data(paths) # condition_data[condition][angles] ~ hdir, hsa, center, post flip, pre flip
    
    # Count the number of sessions and cells
    nCells = len(condition_data["shelter_only"]["hdir_Rayleigh.arrow"]["Rayleigh"]) # Cells are the same for all conditions and angles in one session
    cell_count += nCells
    
    for cell in range(nCells):
        tuned = False # A pointer to check if the cell is tuned to any angle in any condition

        for ci, condition in enumerate(condition_data.keys()):
            rayleigh = 0 # Per condition reset the rayleigh value
            for angle in angle_keys:
                
                #Just the threat zone
                # output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for THREAT ONLY
                # if np.logical_and(output[cell][1] > rayleigh, output[cell][1] > rayleigh_threshold):
                #     rayleigh = output[cell][1]
                #     max_angle_str = regex(angle)
                #     tuned = True
                    
                # Just the shelter
                output = extract_compartment_values(condition_data[condition][angle], column_name="Rayleigh") # Extract the rayleigh values for SHELTER ONLY
                if np.logical_and(output[cell][0] > rayleigh, output[cell][0] > rayleigh_threshold):
                    rayleigh = output[cell][0]
                    max_angle_str = regex(angle)
                    tuned = True

                # # Whole arena rayleigh
                # output = condition_data[condition][angle]["arena_rayleigh"] # Whole arena rayleigh
                # if np.logical_and(output[cell] > rayleigh, output[cell] > rayleigh_threshold):
                #     rayleigh = output[cell]
                #     max_angle_str = regex(angle)
                #     tuned = True
                    
            # For each cell in each session, save the max rayleigh values and corresponding condition | angle combo
            try:
                if tuned:
                    shelter_dict[i][cell][condition] = max_angle_str
                else:
                    shelter_dict[i][cell][condition] = "Not tuned"
            except:
                print(f"Cell {cell} in session {i} in {condition} did not meet threshold")
                
        if not tuned:
            # Track the number of cells that did not meet the threshold
            not_meet_threshold += 1
    

# across session for shelter condition

In [12]:
# Counts across sessions
SC_shelter = []
SC_barrier_pre_flip = []
SC_barrier_post_flip = []
for session in shelter_dict:
    for cell in shelter_dict[session]:
        x = shelter_dict[session][cell]["shelter_only"]
        SC_shelter.append(x)
        y = shelter_dict[session][cell]["barrier_pre_flip"]
        SC_barrier_pre_flip.append(y)
        z = shelter_dict[session][cell]["barrier_post_flip"]
        SC_barrier_post_flip.append(z)

SC_shelter_counts = Counter(SC_shelter)
SC_barrier_pre_flip_counts = Counter(SC_barrier_pre_flip)
SC_barrier_post_flip_counts = Counter(SC_barrier_post_flip)
print("Across session counts")
print(SC_shelter_counts)

SC_shelter_total = 0
SC_barrier_pre_flip_total = 0
SC_barrier_post_flip_total = 0
c1 = sum([i for i in SC_shelter_counts.values()])
c2 = sum([i for i in SC_barrier_pre_flip_counts.values()])
c3 = sum([i for i in SC_barrier_post_flip_counts.values()])
print(f"Cells in shelter_only: {c1}")
print(f"Cells in barrier_pre_flip: {c2}")
print(f"Cells in barrier_post_flip: {c3}")
print(f"Total cells: {cell_count}")

Across session counts
Counter({'Not tuned': 2334, 'hsa': 721, 'h_preflipbar_a': 665, 'h_postflipbar_a': 608, 'hdir': 530})
Cells in shelter_only: 4858
Cells in barrier_pre_flip: 4858
Cells in barrier_post_flip: 4858
Total cells: 4858


In [13]:
# Counts within sessions
SC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
SC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
SC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

for session in shelter_dict:
    for cell in shelter_dict[session]:
        x = shelter_dict[session][cell]["shelter_only"]
        y = shelter_dict[session][cell]["barrier_pre_flip"]
        z = shelter_dict[session][cell]["barrier_post_flip"]
        SC_within_session_counts_shelter[session][x] += 1
        SC_within_session_counts_bar_pre_flip[session][y] += 1
        SC_within_session_counts_bar_post_flip[session][z] += 1

# Single example

In [14]:
# create two subplots that share the same y axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
xcoords = [0, 1, 3, 4]

ax1.set_ylabel('Fraction of cells', fontsize=16)

ax1.set_xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)
ax1.set_ylabel('Fraction of cells', fontsize=16)

# Counts within sessions
TC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

TC_mouse_dict = defaultdict(list)

for session, session_name in zip(threat_dict, session_names):
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
    
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        y = threat_dict[session][cell]["barrier_pre_flip"]
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_within_session_counts_shelter[session][x] += 1
        TC_within_session_counts_bar_pre_flip[session][y] += 1
        TC_within_session_counts_bar_post_flip[session][z] += 1
    
    # Scatter plot points
    y_scatter_values = [
        TC_within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values())
    ]
            
    # Draw lines between scatter points for each session
    ax1.scatter(xcoords, y_scatter_values, color='darkorchid', alpha=0.5)
    ax1.plot(xcoords[:2], y_scatter_values[:2], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 0 and 1
    ax1.plot(xcoords[2:], y_scatter_values[2:], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 3 and 4
    
    TC_mouse_dict[mouse].append(y_scatter_values)
    
    break

ax1.spines[['right', 'top']].set_visible(False)
plt.ylim(0, 0.3)
plt.show()



# plot

In [15]:
# create two subplots that share the same y axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
xcoords = [0, 1, 3, 4]

# THREAT --------------------------------------------------------------------------------------------

# Plot the threat zone only
ax1.bar(xcoords, 
        [TC_barrier_pre_flip_counts['h_preflipbar_a'] / cell_count, 
         TC_barrier_pre_flip_counts['h_postflipbar_a'] / cell_count, 
         TC_barrier_post_flip_counts['h_preflipbar_a'] / cell_count, 
         TC_barrier_post_flip_counts['h_postflipbar_a'] / cell_count],
        color= 'darkorchid',
        alpha = 0.7
        )
ax1.set_xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)
ax1.set_ylabel('Fraction of cells', fontsize=16)

# Counts within sessions
TC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
TC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

TC_mouse_dict = defaultdict(list)

for session, session_name in zip(threat_dict, session_names):
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
    
    for cell in threat_dict[session]:
        x = threat_dict[session][cell]["shelter_only"]
        y = threat_dict[session][cell]["barrier_pre_flip"]
        z = threat_dict[session][cell]["barrier_post_flip"]
        TC_within_session_counts_shelter[session][x] += 1
        TC_within_session_counts_bar_pre_flip[session][y] += 1
        TC_within_session_counts_bar_post_flip[session][z] += 1
    
    # Scatter plot points
    y_scatter_values = [
        TC_within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_pre_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values()),
        TC_within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(TC_within_session_counts_bar_post_flip[session].values())
    ]
        
    ax1.scatter(xcoords, y_scatter_values, color='darkorchid', s=100, alpha=1, marker="")
    
    # Draw lines between scatter points for each session
    ax1.plot(xcoords[:2], y_scatter_values[:2], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 0 and 1
    ax1.plot(xcoords[2:], y_scatter_values[2:], color='darkorchid', alpha=0.5, linewidth = .75)  # Connect points at 3 and 4
    
    TC_mouse_dict[mouse].append(y_scatter_values)

# Apply average mouse information
for mouse in TC_mouse_dict:
    yvals = np.mean(np.array(TC_mouse_dict[mouse]), axis=0)
    ax1.plot(xcoords[:2], yvals[:2], linestyle='-', color="darkorchid", linewidth=3, label=f'{mouse} Average')
    ax1.plot(xcoords[2:], yvals[2:], linestyle='-', color="darkorchid", linewidth=3)
    
# SHELTER --------------------------------------------------------------------------------------------

# Plot the shelter zone only
# Plot the threat zone only
ax2.bar(xcoords, 
        [SC_barrier_pre_flip_counts['h_preflipbar_a'] / cell_count, 
         SC_barrier_pre_flip_counts['h_postflipbar_a'] / cell_count, 
         SC_barrier_post_flip_counts['h_preflipbar_a'] / cell_count, 
         SC_barrier_post_flip_counts['h_postflipbar_a'] / cell_count],
        color= 'cornflowerblue',
        alpha = 0.7
        )
ax2.set_xticks(xcoords, 
           ['Open Edge | Pre flip condition', 
            'Closed Edge | Pre flip condition', 
            'Closed Edge | Post flip condition', 
            'Open Edge | Post flip condition'],
           rotation=10)

# Counts within sessions
SC_within_session_counts_shelter = defaultdict(lambda: defaultdict(int))
SC_within_session_counts_bar_pre_flip = defaultdict(lambda: defaultdict(int))
SC_within_session_counts_bar_post_flip = defaultdict(lambda: defaultdict(int))

SC_mouse_dict = defaultdict(list)

for session, session_name in zip(shelter_dict, session_names):
    
    # Find the mouse for the current session
    mouse = None
    for key, sessions in mice_groups.items():
        if session_name in sessions:
            mouse = key
            break
    
    for cell in shelter_dict[session]:
        x = shelter_dict[session][cell]["shelter_only"]
        y = shelter_dict[session][cell]["barrier_pre_flip"]
        z = shelter_dict[session][cell]["barrier_post_flip"]
        SC_within_session_counts_shelter[session][x] += 1
        SC_within_session_counts_bar_pre_flip[session][y] += 1
        SC_within_session_counts_bar_post_flip[session][z] += 1
    
    # Scatter plot points
    SC_y_scatter_values = [
        SC_within_session_counts_bar_pre_flip[session]['h_preflipbar_a'] / sum(SC_within_session_counts_bar_pre_flip[session].values()),
        SC_within_session_counts_bar_pre_flip[session]['h_postflipbar_a'] / sum(SC_within_session_counts_bar_pre_flip[session].values()),
        SC_within_session_counts_bar_post_flip[session]['h_preflipbar_a'] / sum(SC_within_session_counts_bar_post_flip[session].values()),
        SC_within_session_counts_bar_post_flip[session]['h_postflipbar_a'] / sum(SC_within_session_counts_bar_post_flip[session].values())
    ]
        
    ax2.scatter(xcoords, SC_y_scatter_values, color='cornflowerblue', s=100, alpha=1, marker="")
    
    # Draw lines between scatter points for each session
    ax2.plot(xcoords[:2], SC_y_scatter_values[:2], color='cornflowerblue', alpha=0.5, linewidth = .75)  # Connect points at 0 and 1
    ax2.plot(xcoords[2:], SC_y_scatter_values[2:], color='cornflowerblue', alpha=0.5, linewidth = .75)  # Connect points at 3 and 4
    
    SC_mouse_dict[mouse].append(SC_y_scatter_values)
    
# Apply average mouse information
for mouse in SC_mouse_dict:
    yvals = np.mean(np.array(SC_mouse_dict[mouse]), axis=0)
    ax2.plot(xcoords[:2], yvals[:2], linestyle='-', color="cornflowerblue", linewidth=3, label=f'{mouse} Average')
    ax2.plot(xcoords[2:], yvals[2:], linestyle='-', color="cornflowerblue", linewidth=3)
    
# attach a colour per mouse
# colors = ['red', 'blue', 'green', 'orange', 'purple']
# for mouse, color in zip(mouse_dict, colors):
#     yvals = np.mean(np.array(mouse_dict[mouse]), axis=0)
#     plt.plot(xcoords[:2], yvals[:2], linestyle='-', color=color, linewidth=3, label=f'{mouse} Average')
#     plt.plot(xcoords[2:], yvals[2:], linestyle='-', color=color, linewidth=3)

ax1.spines[['right', 'top']].set_visible(False)
ax2.spines[['right', 'top']].set_visible(False)

plt.show()